In [29]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
import re

import os
from dotenv import load_dotenv
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import ToolCorrectnessMetric
from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
import pandas as pd
from deepeval.models import OllamaModel
from pathlib import Path
from deepeval.evaluate import AsyncConfig


# Walk up from notebook dir until we find the project root containing .env.local
_dir = Path.cwd()
while not (_dir / ".env.local").exists() and _dir != _dir.parent:
    _dir = _dir.parent
env_path = _dir / ".env.local"
print("Using:", env_path, "exists:", env_path.exists())

load_dotenv(env_path, override=True)

CLOUD_MODEL_BASE_URL = os.getenv("CLOUD_MODEL_BASE_URL")
LOCAL_MODEL_BASE_URL = os.getenv("LOCAL_MODEL_BASE_URL")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")

Using: /Users/michelecandolfo/Documents/workspaces/DeepEval/ai-engineering-portfolio/.env.local exists: True


In [30]:
llm = ChatOllama(
    base_url=CLOUD_MODEL_BASE_URL,
    model="qwen3.5:cloud",
    temperature=0.3,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},  
)

#### Creating an AI Agent with specific tools

In [31]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(backend="api", region="us-en", safesearch="moderate", max_results=5)


@tool
def add_numbers(a: int, b: int) -> int:
    "Add two numbers and return results."
    return int(a) + int(b)

@tool
def subtract_numbers(a: int, b: int) -> int:
    "Subtract two numbers and return results."
    return int(a) - int(b)

tools = [add_numbers, subtract_numbers, search_tool]

# Bind tools to the LLM so it knows which tools are available
llm_with_tools = llm.bind_tools(tools)
agent = create_agent(llm_with_tools, tools)

response = agent.invoke({"messages": [("user", "What is the sum of 20 and 40?")]})

# Print message flow
for msg in response["messages"]:
    if msg.type == "tool":
        print(f"[tool] {msg.name} -> {msg.content[:100]}")
    elif hasattr(msg, "tool_calls") and msg.tool_calls:
        calls = ", ".join(tc["name"] for tc in msg.tool_calls)
        print(f"[ai] calling tools: {calls}")
    elif hasattr(msg, "content") and msg.content:
        print(f"[{msg.type}] {msg.content[:200]}")

[human] What is the sum of 20 and 40?
[ai] calling tools: add_numbers
[tool] add_numbers -> 60
[ai] The sum of 20 and 40 is **60**.


In [32]:
def query_ai_agent(question):
    """Run the agent and extract tool calls + final answer."""
    response = agent.invoke({"messages": [("user", question)]})

    # Extract tool calls from message history
    tool_names = []
    tool_inputs = []
    for msg in response["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_names.append(tc["name"])
                tool_inputs.append(tc["args"])

    # Get the final AI answer
    final_answer = ""
    for msg in reversed(response["messages"]):
        if msg.type == "ai" and msg.content and not msg.tool_calls:
            final_answer = msg.content
            break

    return response, tool_names, tool_inputs, final_answer

In [34]:
response, tool_names, tool_inputs, final_answer = query_ai_agent("Who is the president of USA in 2026, just give me the name")

print("Final answer:", final_answer)
print("Tools called:", tool_names)
print("Tool inputs:", tool_inputs)

Final answer: Donald Trump
Tools called: ['duckduckgo_search']
Tool inputs: [{'query': 'USA president 2026'}]


#### Testing the AI Agent 

In [35]:
from deepeval.test_case import ToolCall

test_data = [
    {
        "input": "What is sum of 20 and 40?",
        "expected_output": "60",
        "tool_called": [
            ToolCall(
                name="add_numbers",
                #tool_input={"a": 20, "b": 40},
                #tool_output="60"
            )
        ]
    },
    {
        "input": "Who is the president of USA in 2025, just give me the name",
        "expected_output": "Donald Trump",
        "tool_called": [
            ToolCall(
                name="duckduckgo_search",
                
            )
        ]
    },

]

In [36]:
test_data

[{'input': 'What is sum of 20 and 40?',
  'expected_output': '60',
  'tool_called': [ToolCall(
       name="add_numbers"
   )]},
 {'input': 'Who is the president of USA in 2025, just give me the name',
  'expected_output': 'Donald Trump',
  'tool_called': [ToolCall(
       name="duckduckgo_search"
   )]}]

In [37]:
response, tool_names, tool_inputs, final_answer = query_ai_agent(test_data[0]["input"])

print("Final answer:", final_answer)
print("Tools called:", tool_names)
print("Tool inputs:", tool_inputs)

Final answer: The sum of 20 and 40 is **60**.
Tools called: ['add_numbers']
Tool inputs: [{'a': 20, 'b': 40}]


In [38]:
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import ToolCorrectnessMetric
from deepeval.dataset import EvaluationDataset
from deepeval import evaluate

test_cases = []

for testcase in test_data:
    response, tool_names, tool_inputs, final_answer = query_ai_agent(testcase["input"])

    # Build tools_called list from all tools the agent invoked
    tools_called = [ToolCall(name=name) for name in tool_names]

    test_case = LLMTestCase(
        input=testcase["input"],
        actual_output=final_answer,
        tools_called=tools_called,
        expected_tools=testcase["tool_called"],
    )
    test_cases.append(test_case)

In [39]:
test_cases

[LLMTestCase(input='What is sum of 20 and 40?', actual_output='The sum of 20 and 40 is **60**.', expected_output=None, context=None, retrieval_context=None, additional_metadata=None, tools_called=[ToolCall(
     name="add_numbers"
 )], comments=None, expected_tools=[ToolCall(
     name="add_numbers"
 )], token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None),
 LLMTestCase(input='Who is the president of USA in 2025, just give me the name', actual_output='Donald Trump', expected_output=None, context=None, retrieval_context=None, additional_metadata=None, tools_called=[ToolCall(
     name="duckduckgo_search"
 )], comments=None, expected_tools=[ToolCall(
     name="duckduckgo_search"
 )], token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None)]

In [40]:
metrics = ToolCorrectnessMetric()
for testcase in test_cases:
    
    metrics.measure(test_case=testcase)
    print(metrics.score)
    print(metrics.reason)
    print(metrics.expected_tools)

Output()

Output()

1.0
[
	 Tool Calling Reason: All expected tools ['add_numbers'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]

[ToolCall(
    name="add_numbers"
)]


1.0
[
	 Tool Calling Reason: All expected tools ['duckduckgo_search'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]

[ToolCall(
    name="duckduckgo_search"
)]
